# Building MCP Servers with FastMCP

This notebook teaches you how to build Model Context Protocol (MCP) servers using the FastMCP Python framework, covering all three server primitives: **tools**, **resources**, and **prompts**.

## What is MCP?

The **Model Context Protocol (MCP)** is a universal standard for connecting AI applications to external tools and data. Instead of building custom integrations for every AI platform, you build an MCP server once and it works with any MCP-compatible host (Claude Desktop, VS Code, Cursor, OpenAI, and more).

Think of it like **USB-C for AI** — one standard connector for all devices.

### The Three Server Primitives

Every MCP server can expose three types of capabilities:

| Primitive | Purpose | Examples |
|-----------|---------|----------|
| **Tools** | Executable functions the AI can call | API calls, calculations, database queries |
| **Resources** | Read-only data sources for context | File contents, config data, DB records |
| **Prompts** | Reusable templates for LLM interactions | System prompts, few-shot examples |

### What is FastMCP?

[FastMCP](https://github.com/jlowin/fastmcp) is a Python framework that makes building MCP servers simple. You define tools, resources, and prompts with decorators — FastMCP handles the protocol, JSON schemas, and transport automatically.

## Setup

In [ ]:
%pip install -q fastmcp

In [ ]:
from fastmcp import FastMCP, Client

## Part 1: Tools — Executable Functions

Tools are the most commonly used MCP primitive. They let an AI model call functions in your server to perform actions or retrieve computed data.

With FastMCP, you define tools using the `@mcp.tool` decorator. FastMCP automatically:
- Uses the **function name** as the tool name
- Uses the **docstring** as the tool description
- Generates a **JSON schema** from the type hints for the tool's parameters

In [ ]:
# Create an MCP server instance
server = FastMCP("Math Server")


@server.tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


@server.tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b


@server.tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from Celsius to Fahrenheit."""
    return (celsius * 9 / 5) + 32


print("Server created with 3 tools!")

### Testing Tools with the In-Memory Client

FastMCP provides a `Client` class that can connect directly to a server instance in memory — no network or separate process needed. This is perfect for testing and learning.

In Jupyter notebooks, you can use `await` directly at the top level thanks to IPython's autoawait feature.

In [ ]:
async with Client(server) as client:
    # List all available tools
    tools = await client.list_tools()
    print("Available tools:")
    for tool in tools:
        print(f"  - {tool.name}: {tool.description}")
    print()

In [ ]:
async with Client(server) as client:
    # Call the add tool
    result = await client.call_tool("add", {"a": 15, "b": 27})
    print(f"add(15, 27) = {result[0].text}")

    # Call the multiply tool
    result = await client.call_tool("multiply", {"a": 6, "b": 7})
    print(f"multiply(6, 7) = {result[0].text}")

    # Call the celsius_to_fahrenheit tool
    result = await client.call_tool("celsius_to_fahrenheit", {"celsius": 100.0})
    print(f"celsius_to_fahrenheit(100.0) = {result[0].text}")

### Tools with Complex Return Types

Tools can return complex data types like dictionaries and lists. FastMCP automatically serializes them.

In [ ]:
import json

product_server = FastMCP("Product Server")

# Simulated product database
PRODUCTS = [
    {"id": 1, "name": "Wireless Mouse", "price": 29.99, "category": "electronics"},
    {"id": 2, "name": "Mechanical Keyboard", "price": 89.99, "category": "electronics"},
    {"id": 3, "name": "USB-C Hub", "price": 49.99, "category": "electronics"},
    {"id": 4, "name": "Standing Desk", "price": 399.99, "category": "furniture"},
    {"id": 5, "name": "Monitor Light Bar", "price": 59.99, "category": "accessories"},
]


@product_server.tool
def search_products(query: str, max_results: int = 3) -> list[dict]:
    """Search for products by name. Returns matching products with id, name, price, and category."""
    results = [p for p in PRODUCTS if query.lower() in p["name"].lower()]
    return results[:max_results]


@product_server.tool
def get_product_by_id(product_id: int) -> dict:
    """Get detailed information about a specific product by its ID."""
    for p in PRODUCTS:
        if p["id"] == product_id:
            return p
    return {"error": f"Product with id {product_id} not found"}


# Test it
async with Client(product_server) as client:
    result = await client.call_tool("search_products", {"query": "keyboard"})
    print("Search for 'keyboard':")
    print(result[0].text)
    print()

    result = await client.call_tool("get_product_by_id", {"product_id": 3})
    print("Get product #3:")
    print(result[0].text)

## Part 2: Resources — Read-Only Data Sources

Resources provide contextual data that AI models can read. Unlike tools (which perform actions), resources are **read-only** — they expose data the model might need to answer questions.

Resources are identified by URIs. FastMCP supports two types:
- **Static resources**: Fixed URI like `config://app` — always returns the same data
- **Resource templates**: Parameterized URI like `users://{user_id}` — returns different data based on the parameter

In [ ]:
resource_server = FastMCP("Resource Demo")


# Static resource — fixed URI, always returns the same data
@resource_server.resource("config://app")
def get_app_config() -> dict:
    """Application configuration settings."""
    return {
        "app_name": "MyApp",
        "version": "2.1.0",
        "environment": "production",
        "max_users": 1000,
    }


# Static resource — company FAQ
@resource_server.resource("docs://faq")
def get_faq() -> str:
    """Frequently asked questions."""
    return """Q: What are your business hours?
A: We are open Monday-Friday, 9am-5pm EST.

Q: How do I reset my password?
A: Click 'Forgot Password' on the login page.

Q: What is your refund policy?
A: Full refund within 30 days of purchase."""


# Resource template — parameterized URI
@resource_server.resource("users://{user_id}/profile")
def get_user_profile(user_id: str) -> dict:
    """User profile information."""
    # Simulated user database
    users = {
        "alice": {"name": "Alice Smith", "role": "admin", "email": "alice@example.com"},
        "bob": {"name": "Bob Jones", "role": "user", "email": "bob@example.com"},
    }
    return users.get(user_id, {"error": f"User '{user_id}' not found"})


print("Resource server created with 2 static resources and 1 template!")

In [ ]:
async with Client(resource_server) as client:
    # List all available resources
    resources = await client.list_resources()
    print("Available resources:")
    for r in resources:
        print(f"  - {r.uri}: {r.description}")
    print()

    # List resource templates
    templates = await client.list_resource_templates()
    print("Resource templates:")
    for t in templates:
        print(f"  - {t.uriTemplate}: {t.description}")
    print()

In [ ]:
async with Client(resource_server) as client:
    # Read the static config resource
    content = await client.read_resource("config://app")
    print("App config:")
    print(content[0].text)
    print()

    # Read the FAQ resource
    content = await client.read_resource("docs://faq")
    print("FAQ:")
    print(content[0].text)
    print()

    # Read a parameterized user profile
    content = await client.read_resource("users://alice/profile")
    print("Alice's profile:")
    print(content[0].text)

## Part 3: Prompts — Reusable Templates

Prompts are pre-built templates that structure how an AI model should approach a task. They can include:
- A single user message (returned as a string)
- A multi-turn conversation (returned as a list of messages)
- Dynamic parameters filled in at request time

Prompts are useful for standardizing common interactions — like code review, debugging, or summarization — across different AI clients.

In [ ]:
from fastmcp.prompts import Message

prompt_server = FastMCP("Prompt Demo")


# Simple prompt — returns a string (automatically wrapped as a user message)
@prompt_server.prompt
def explain_concept(topic: str) -> str:
    """Generate a prompt asking for an explanation of a topic."""
    return f"Please explain the concept of '{topic}' in simple terms that a beginner could understand. Include a practical example."


# Multi-turn prompt — returns a list of messages to set up a conversation
@prompt_server.prompt
def code_review(code: str, language: str = "python") -> list[Message]:
    """Set up a code review conversation."""
    return [
        Message(
            role="user",
            content=f"Please review the following {language} code for bugs, style issues, and potential improvements:\n\n```{language}\n{code}\n```",
        ),
        Message(
            role="assistant",
            content="I'll review this code carefully. Let me analyze it for correctness, style, and potential improvements.",
        ),
    ]


# Debugging helper prompt
@prompt_server.prompt
def debug_error(error_message: str, context: str = "") -> list[Message]:
    """Set up a debugging session for an error."""
    user_content = f"I encountered this error:\n\n```\n{error_message}\n```"
    if context:
        user_content += f"\n\nAdditional context: {context}"
    return [
        Message(role="user", content=user_content),
    ]


print("Prompt server created with 3 prompts!")

In [ ]:
async with Client(prompt_server) as client:
    # List all available prompts
    prompts = await client.list_prompts()
    print("Available prompts:")
    for p in prompts:
        print(f"  - {p.name}: {p.description}")
    print()

    # Get a simple prompt
    result = await client.get_prompt("explain_concept", {"topic": "Model Context Protocol"})
    print("explain_concept prompt:")
    for msg in result.messages:
        print(f"  [{msg.role}]: {msg.content.text}")
    print()

    # Get a multi-turn prompt
    result = await client.get_prompt(
        "code_review",
        {"code": "def add(a, b):\n    return a + b", "language": "python"},
    )
    print("code_review prompt:")
    for msg in result.messages:
        print(f"  [{msg.role}]: {msg.content.text[:80]}...")

## Part 4: Putting It All Together

A real MCP server typically combines all three primitives. Here is a complete example of a **Company Assistant** server that exposes tools, resources, and prompts together.

In [ ]:
from fastmcp.prompts import Message

assistant_server = FastMCP("Company Assistant")

# --- TOOLS ---

EMPLOYEES = {
    "E001": {"name": "Alice Johnson", "dept": "Engineering", "title": "Senior Developer"},
    "E002": {"name": "Bob Williams", "dept": "Marketing", "title": "Marketing Manager"},
    "E003": {"name": "Carol Davis", "dept": "Engineering", "title": "Tech Lead"},
}


@assistant_server.tool
def lookup_employee(employee_id: str) -> dict:
    """Look up an employee by their ID. Returns name, department, and title."""
    return EMPLOYEES.get(employee_id, {"error": f"Employee {employee_id} not found"})


@assistant_server.tool
def list_department_members(department: str) -> list[dict]:
    """List all employees in a given department."""
    return [
        {"id": eid, **info}
        for eid, info in EMPLOYEES.items()
        if info["dept"].lower() == department.lower()
    ]


# --- RESOURCES ---


@assistant_server.resource("company://handbook")
def company_handbook() -> str:
    """The company employee handbook."""
    return """Company Handbook Summary:
- Work hours: Flexible, core hours 10am-3pm
- PTO: 20 days per year
- Remote work: Hybrid (3 days office, 2 days remote)
- Benefits: Health, dental, vision, 401k matching"""


@assistant_server.resource("company://departments/{dept_name}")
def department_info(dept_name: str) -> dict:
    """Information about a specific department."""
    departments = {
        "engineering": {"head": "Carol Davis", "headcount": 25, "budget": "$2.5M"},
        "marketing": {"head": "Bob Williams", "headcount": 10, "budget": "$1.2M"},
    }
    return departments.get(dept_name.lower(), {"error": f"Department '{dept_name}' not found"})


# --- PROMPTS ---


@assistant_server.prompt
def onboarding_checklist(employee_name: str, department: str) -> str:
    """Generate an onboarding checklist for a new employee."""
    return f"""Create a first-week onboarding checklist for {employee_name} joining the {department} department.
Include:
- IT setup tasks
- Required meetings
- Documentation to review
- Team introductions"""


print("Company Assistant server created!")

In [ ]:
async with Client(assistant_server) as client:
    # Discover all capabilities
    tools = await client.list_tools()
    resources = await client.list_resources()
    templates = await client.list_resource_templates()
    prompts = await client.list_prompts()

    print("=== Company Assistant Server Capabilities ===")
    print(f"\nTools ({len(tools)}):")
    for t in tools:
        print(f"  - {t.name}: {t.description}")

    print(f"\nResources ({len(resources)}):")
    for r in resources:
        print(f"  - {r.uri}: {r.description}")

    print(f"\nResource Templates ({len(templates)}):")
    for t in templates:
        print(f"  - {t.uriTemplate}: {t.description}")

    print(f"\nPrompts ({len(prompts)}):")
    for p in prompts:
        print(f"  - {p.name}: {p.description}")

In [ ]:
async with Client(assistant_server) as client:
    # Use a tool: look up an employee
    result = await client.call_tool("lookup_employee", {"employee_id": "E001"})
    print("Employee E001:", result[0].text)
    print()

    # Use a tool: list Engineering members
    result = await client.call_tool("list_department_members", {"department": "Engineering"})
    print("Engineering team:", result[0].text)
    print()

    # Read a resource: company handbook
    content = await client.read_resource("company://handbook")
    print("Handbook:", content[0].text)
    print()

    # Read a resource template: department info
    content = await client.read_resource("company://departments/engineering")
    print("Engineering dept:", content[0].text)
    print()

    # Get a prompt: onboarding checklist
    prompt = await client.get_prompt(
        "onboarding_checklist",
        {"employee_name": "Dave", "department": "Engineering"},
    )
    print("Onboarding prompt:", prompt.messages[0].content.text)

## How This Connects to AI Applications

In this notebook, we used FastMCP's in-memory `Client` to test our servers directly. In production, MCP servers run over a **transport layer** and are accessed by AI applications:

| Transport | How It Works | Use Case |
|-----------|-------------|----------|
| **stdio** | Local process communication | Claude Desktop, local tools |
| **Streamable HTTP** | Remote HTTP server with SSE | OpenAI, cloud deployments |

In the next notebook, we will run our server over HTTP and connect it to the **OpenAI Responses API**.

## Summary

- **MCP** is a universal protocol for connecting AI to external tools and data — build once, connect everywhere.
- **FastMCP** makes building MCP servers simple with Python decorators.
- The three primitives are:
  - **Tools** (`@server.tool`): Executable functions the AI can call. Type hints auto-generate the JSON schema.
  - **Resources** (`@server.resource("uri")`): Read-only data sources identified by URIs. Support both static and parameterized templates.
  - **Prompts** (`@server.prompt`): Reusable conversation templates that can return strings or multi-turn message lists.
- The **in-memory `Client`** lets you test servers without any network setup — ideal for development and learning.
- MCP servers run over transport layers (stdio or HTTP) to connect to real AI applications.